# Saher × ComfyUI على Kaggle Notebooks

هذا الدفتر مخصص للتجربة التفاعلية داخل Kaggle Notebook. يشغّل ComfyUI محلياً على loopback داخل جلسة الدفتر، ويرسل وصف الفيديو عبر محول Saher، ثم يعرض MP4 ضمن مخرجات الخلايا. **لن يفتح صفحة ويب مستقلة ولن ينشئ رابطاً عاماً أو Tunnel.**

> **التوفر:** توثيق Kaggle يذكر جلسة تنفيذ تصل إلى 12 ساعة وGPU مثل P100 أو T4×2 بحسب البيئة، لكن توفر GPU محدود وقد يتطلب انتظاراً، وقد تتغير الموارد. الدفتر يفحص بطاقة GPU والذاكرة قبل تنزيل الأوزان. [توثيق Kaggle](https://www.kaggle.com/docs/notebooks)

نموذج Wan2.1 T2V-1.3B يتطلب تقريباً 8 GB VRAM بحسب الوثائق الرسمية؛ ذلك حد إرشادي وليس ضمان تشغيل على كل بطاقة. التجربة مضبوطة على 832×480 و17 إطاراً و20 خطوة. نزّل ملفات النموذج عدة غيغابايت، ولذلك فعّل **Internet** من إعدادات الدفتر. ملفات `/kaggle/working` مؤقتة حتى حفظ نسخة/مخرجات الدفتر أو تنزيل الفيديو.

**البدء:** أنشئ Kaggle Notebook جديداً واستورد ملف هذا الدفتر. من Settings اختر Accelerator: GPU، وفعّل Internet، ثم شغّل الخلايا بالترتيب. إذا لم يتوفر GPU أو الإنترنت، أوقف ولا تكمل التنزيل.

## 1. فحص Python وGPU قبل التنزيل


In [ ]:
import sys, torch
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('لم يتوفر GPU في هذه الجلسة. اختر Runtime > Change runtime type > GPU، أو حاول لاحقاً؛ التوفر المجاني غير مضمون.')
props = torch.cuda.get_device_properties(0)
vram_gib = props.total_memory / 1024**3
print('GPU:', props.name, '| VRAM GiB:', round(vram_gib, 1))
if vram_gib < 8:
    raise RuntimeError(f'الذاكرة المعروضة {vram_gib:.1f} GiB أقل من حد ComfyUI الموصى به لهذا المسار (نحو 8 GiB). أوقف التجربة لتجنب تنزيل الأوزان ثم الفشل.')
print('GPU check passed. This is only a hardware check, not a generation benchmark.')

## 2. تنزيل ComfyUI وفرع Saher التجريبي وتثبيت الاعتماديات

تأكد أن خيار Internet مفعّل من إعدادات Kaggle Notebook؛ نحتاجه لاستنساخ GitHub وتنزيل الأوزان من Hugging Face.

In [ ]:
%cd /kaggle/working
!git clone --depth 1 https://github.com/Comfy-Org/ComfyUI.git
!git clone --depth 1 --branch feature/saher-mvp-foundation https://github.com/jonbander/saher-video-colab.git
%pip install -r /kaggle/working/ComfyUI/requirements.txt
%pip install huggingface_hub
%pip install -e /kaggle/working/saher-video-colab

## 3. تنزيل أوزان Wan2.1 الرسمية

تنزّل الخلية نموذج T2V-1.3B بنسخة fp16 ومشفّر نص fp8 وVAE إلى مساحة العمل المؤقتة. تتطلب التنزيلات عدة غيغابايت. يوصي الدليل بدقة 480p لهذا الطراز. [دليل ComfyUI الرسمي](https://docs.comfy.org/tutorials/video/wan/wan-video)

In [ ]:
from pathlib import Path
import shutil
from huggingface_hub import HfApi, hf_hub_download

COMFY = Path('/kaggle/working/ComfyUI')
MODEL_REPO = 'Comfy-Org/Wan_2.1_ComfyUI_repackaged'
revision = HfApi().model_info(MODEL_REPO).sha
print('Pinned model repository revision for this run:', revision)

files = {
    'split_files/diffusion_models/wan2.1_t2v_1.3B_fp16.safetensors': COMFY/'models/diffusion_models/wan2.1_t2v_1.3B_fp16.safetensors',
    'split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors': COMFY/'models/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors',
    'split_files/vae/wan_2.1_vae.safetensors': COMFY/'models/vae/wan_2.1_vae.safetensors',
}
for repo_file, target in files.items():
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        print(f'Downloading {Path(repo_file).name} ...')
        cached = Path(hf_hub_download(repo_id=MODEL_REPO, filename=repo_file, revision=revision, local_dir=str(COMFY/'models')))
        shutil.move(cached, target)
    print(f'{target.name}: {target.stat().st_size/1024**3:.2f} GiB')
print('Model files ready.')

## 4. إنشاء workflow API متوافق مع Saher

يستخدم workflow الأصلي لـ Wan2.1 T2V في ComfyUI مع دقة 832×480 و17 إطاراً. لا نفتح صفحة ComfyUI؛ نتصل بواجهة API المحلية من خلايا الدفتر.

In [ ]:
import json

workflow = {
  '37': {'class_type':'UNETLoader','inputs':{'unet_name':'wan2.1_t2v_1.3B_fp16.safetensors','weight_dtype':'default'}},
  '38': {'class_type':'CLIPLoader','inputs':{'clip_name':'umt5_xxl_fp8_e4m3fn_scaled.safetensors','type':'wan','device':'default'}},
  '39': {'class_type':'VAELoader','inputs':{'vae_name':'wan_2.1_vae.safetensors'}},
  '6': {'class_type':'CLIPTextEncode','inputs':{'clip':['38',0],'text':'A cinematic shot of a small boat crossing a calm lake at sunrise, slow camera movement, soft golden light'}},
  '7': {'class_type':'CLIPTextEncode','inputs':{'clip':['38',0],'text':'blurry, distorted, low quality, text, watermark'}},
  '40': {'class_type':'EmptyHunyuanLatentVideo','inputs':{'width':832,'height':480,'length':17,'batch_size':1}},
  '48': {'class_type':'ModelSamplingSD3','inputs':{'model':['37',0],'shift':8}},
  '3': {'class_type':'KSampler','inputs':{'model':['48',0],'positive':['6',0],'negative':['7',0],'latent_image':['40',0],'seed':20261025,'steps':20,'cfg':6,'sampler_name':'uni_pc','scheduler':'simple','denoise':1.0}},
  '8': {'class_type':'VAEDecode','inputs':{'samples':['3',0],'vae':['39',0]}},
  '49': {'class_type':'CreateVideo','inputs':{'images':['8',0],'fps':16}},
  '50': {'class_type':'SaveVideo','inputs':{'video':['49',0],'filename_prefix':'video/SaherTest','format':'auto','codec':'auto'}}
}
workflow_path = Path('/kaggle/working/saher-wan21-api.json')
workflow_path.write_text(json.dumps(workflow, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved API workflow:', workflow_path)

## 5. تشغيل ComfyUI محلياً داخل جلسة Kaggle


In [ ]:
import subprocess, time, requests
from pathlib import Path

log_file = open('/kaggle/working/comfyui.log', 'w')
comfy_process = subprocess.Popen([sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', '8188'], cwd=str(COMFY), stdout=log_file, stderr=subprocess.STDOUT)
base = 'http://127.0.0.1:8188'
for attempt in range(120):
    if comfy_process.poll() is not None:
        raise RuntimeError('ComfyUI exited early. Inspect /kaggle/working/comfyui.log. Last log lines:\n' + Path('/kaggle/working/comfyui.log').read_text(errors='replace')[-5000:])
    try:
        health = requests.get(base + '/system_stats', timeout=3)
        if health.ok:
            print('ComfyUI local API ready on loopback; it is not exposed through a public tunnel.')
            print('Reported device:', health.json().get('system',{}).get('pytorch_version','PyTorch'))
            break
    except requests.RequestException:
        time.sleep(2)
else:
    raise RuntimeError('ComfyUI startup timed out. Inspect /kaggle/working/comfyui.log.')

## 6. إرسال الوصف من الدفتر وعرض الفيديو الناتج

غيّر قيمة `PROMPT` في الخلية التالية إلى وصفك. يستدعي Saher وComfyUI محلياً داخل الجلسة، ثم يعرض الفيديو هنا؛ لا توجد صفحة ويب خارجية.

In [ ]:
import os, time
from fastapi.testclient import TestClient

PROMPT = 'A cinematic shot of a small boat crossing a calm lake at sunrise, slow camera movement, soft golden light'  # عدّل هذا الوصف قبل التشغيل

os.environ['COMFYUI_BASE_URL'] = 'http://127.0.0.1:8188'
os.environ['COMFYUI_WORKFLOW_PATH'] = str(workflow_path)
os.environ['COMFYUI_PROMPT_NODE_ID'] = '6'
os.environ['COMFYUI_PROMPT_INPUT'] = 'text'

# Ensure a fresh module reads the workflow path set above.
import importlib, saher_video.app
importlib.reload(saher_video.app)
client = TestClient(saher_video.app.app)
prompt = PROMPT.strip()
if len(prompt) < 3:
    raise ValueError('اكتب وصفاً من 3 أحرف على الأقل في متغير PROMPT.')
workflow['6']['inputs']['text'] = prompt
workflow_path.write_text(json.dumps(workflow, ensure_ascii=False, indent=2), encoding='utf-8')

submitted = client.post('/api/v1/jobs', json={'prompt': prompt})
assert submitted.status_code == 202, submitted.text
job_id = submitted.json()['job_id']
print('Saher job id:', job_id)

job = None
for attempt in range(180):  # Up to 30 minutes; Kaggle runtime may terminate sooner.
    response = client.get(f'/api/v1/jobs/{job_id}')
    if response.status_code != 200:
        raise RuntimeError(response.text)
    job = response.json()
    if attempt % 3 == 0:
        print(f"Elapsed ~{attempt*10}s | state={job['state']}")
    if job['state'] == 'completed':
        break
    if job['state'] == 'failed':
        raise RuntimeError(job.get('error') or 'ComfyUI generation failed')
    if job['state'] == 'unknown':
        raise RuntimeError('ComfyUI no longer has this job in its queue/history')
    time.sleep(10)
else:
    raise TimeoutError('انتهى الانتظار بعد 30 دقيقة؛ افحص /kaggle/working/comfyui.log إذا كانت الجلسة ما زالت تعمل.')

videos = [item for item in job['outputs'] if item['content_type'].startswith('video/')]
if not videos:
    raise RuntimeError(f'No video output returned. Check that the workflow has a SaveVideo node. Outputs: {job["outputs"]}')
asset = client.get(videos[0]['url'])
assert asset.ok, asset.text[:1000]
output_path = Path('/kaggle/working/saher-kaggle-test.mp4')
output_path.write_bytes(asset.content)
print(f'تم حفظ الفيديو مؤقتاً: {output_path} ({output_path.stat().st_size/1024**2:.1f} MiB)')
from IPython.display import FileLink
display(FileLink(str(output_path)))
from IPython.display import Video, display
display(Video(str(output_path), embed=True, width=480))

## 7. حفظ الفيديو وإيقاف ComfyUI

يظهر الفيديو مضمّناً أعلاه. استخدم رابط التنزيل الناتج من الخلية لحفظه على جهازك، أو أضف `/kaggle/working/saher-kaggle-test.mp4` إلى Outputs عند حفظ نسخة الدفتر. شغّل الخلية التالية لإنهاء ComfyUI.

In [ ]:
if 'comfy_process' in globals() and comfy_process.poll() is None:
    comfy_process.terminate()
    try:
        comfy_process.wait(timeout=20)
    except subprocess.TimeoutExpired:
        comfy_process.kill()
    print('ComfyUI stopped.')
else:
    print('ComfyUI is not running.')

### المصادر

- [Kaggle Notebooks: GPU مجاني، الإتاحة ومواصفات الجلسة](https://www.kaggle.com/docs/notebooks)
- [ComfyUI: Wan2.1 T2V-1.3B workflow وملفات النموذج](https://docs.comfy.org/tutorials/video/wan/wan-video)
- [قالب Wan2.1 الرسمي في ComfyUI](https://github.com/Comfy-Org/workflow_templates/blob/main/templates/text_to_video_wan.json)
- [بطاقة Wan2.1 T2V-1.3B ورخصة Apache-2.0](https://huggingface.co/Wan-AI/Wan2.1-T2V-1.3B)
- [Saher — فرع التجربة](https://github.com/jonbander/saher-video-colab/tree/feature/saher-mvp-foundation)